# Dataset Visualizations
Publication-quality figure of MADOS, SpaceNet7-CD, and PASTIS for thesis use.

In [ ]:
import os
import numpy as np
import rasterio
import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap, BoundaryNorm
from pathlib import Path
from glob import glob

warnings.filterwarnings('ignore')

DATASET_ROOT = Path('/dss/dsstbyfs02/pn49cu/pn49cu-dss-0001/pangaea-datasets')
OUT_DIR      = Path('/dss/dsshome1/08/di54zop/pangaea_experiments')
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        9,
    'axes.titlesize':   10,
    'axes.titleweight': 'bold',
})
print('Ready.')

In [ ]:
def to_rgb(img_chw, r, g, b, pct_lo=2, pct_hi=98):
    """Band selection + percentile stretch → float32 H×W×3 in [0,1]."""
    rgb = np.stack([img_chw[r], img_chw[g], img_chw[b]], axis=-1).astype(np.float32)
    lo, hi = np.percentile(rgb, pct_lo), np.percentile(rgb, pct_hi)
    return np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)

def make_cmap(colors):
    n = len(colors)
    cmap = ListedColormap(colors)
    norm = BoundaryNorm(np.arange(n + 1) - 0.5, n)
    return cmap, norm

def ax_img(ax, img, title='', titlepad=4):
    ax.imshow(img, interpolation='nearest')
    ax.set_title(title, pad=titlepad)
    ax.axis('off')

def ax_mask(ax, mask, cmap, norm, title='', titlepad=4):
    ax.imshow(mask, cmap=cmap, norm=norm, interpolation='nearest')
    ax.set_title(title, pad=titlepad)
    ax.axis('off')

def legend_ax(ax, labels, colors, ncol=1, fontsize=8):
    ax.axis('off')
    patches = [mpatches.Patch(facecolor=c, edgecolor='0.3', linewidth=0.5, label=l)
               for l, c in zip(labels, colors)]
    ax.legend(handles=patches, loc='center left', frameon=False,
              fontsize=fontsize, ncol=ncol,
              handlelength=1.1, handleheight=1.1, handletextpad=0.5,
              labelspacing=0.35, borderpad=0)

print('Helpers defined.')

## MADOS — Marine Debris & Oil Spill Segmentation

In [ ]:
MADOS_ROOT    = DATASET_ROOT / 'mados'
MADOS_TILE    = 'Scene_96'
MADOS_CROP    = '1.tif'

MADOS_CLASSES = [
    'Marine Debris', 'Dense Sargassum', 'Sparse Floating Algae',
    'Natural Organic Material', 'Ship', 'Oil Spill', 'Marine Water',
    'Sediment-Laden Water', 'Foam', 'Turbid Water', 'Shallow Water',
    'Waves & Wakes', 'Oil Platform', 'Jellyfish', 'Sea snot',
]
MADOS_COLORS = [
    '#e41a1c', '#ff7f00', '#ffff33', '#a65628', '#f781bf',  # 0-4
    '#1a1a1a', '#377eb8', '#a6cee3', '#ffffff', '#4daf4a',  # 5-9
    '#00bfff', '#984ea3', '#8B4513', '#17becf', '#bcbd22',  # 10-14
]

# Load all spectral bands (11 bands: B1,B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12)
tile_dir  = MADOS_ROOT / MADOS_TILE
all_bands = sorted(glob(str(tile_dir / '*' / f'*L2R_rhorc*_{MADOS_CROP}')),
                   key=lambda p: int(p.split('_')[-2]))

imgs = []
for bp in all_bands:
    upscale = int(os.path.basename(os.path.dirname(bp))) // 10
    with rasterio.open(bp) as src:
        b = src.read(1,
                     out_shape=(src.height * upscale, src.width * upscale),
                     resampling=rasterio.enums.Resampling.nearest).astype(np.float32)
    imgs.append(b)
mados_img = np.stack(imgs)   # (11, H, W)

with rasterio.open(str(tile_dir / '10' / f'{MADOS_TILE}_L2R_cl_{MADOS_CROP}')) as src:
    mados_gt = src.read(1).astype(np.int64) - 1   # 0-based; -1 = ignore

# RGB: B4(idx3)=Red, B3(idx2)=Green, B2(idx1)=Blue
mados_rgb     = to_rgb(mados_img, r=3, g=2, b=1)
mados_gt_disp = np.ma.masked_where(mados_gt == -1, mados_gt)
mados_present = sorted(set(np.unique(mados_gt)) - {-1})

print(f'Shape: {mados_img.shape},  classes: {[MADOS_CLASSES[c] for c in mados_present]}')

## SpaceNet7-CD — Building Change Detection

In [ ]:
SN7_ROOT = DATASET_ROOT / 'spacenet7cd'
SN7_AOI  = 'L15-1296E-1198N_5184_3399_13'
SN7_Y1, SN7_M1 = 2018, 1
SN7_Y2, SN7_M2 = 2020, 1

# Planet imagery: 4 bands (R, G, B, NIR) → use first 3 for RGB
CROP = np.s_[576:976, 576:976]   # 400×400 subregion with densest change

def load_sn7_img(aoi, y, m):
    p = SN7_ROOT / 'train' / aoi / 'images_masked' / \
        f'global_monthly_{y}_{m:02d}_mosaic_{aoi}.tif'
    with rasterio.open(str(p)) as src:
        return src.read()[:3, CROP[0], CROP[1]].astype(np.float32)  # (3, H, W)

def load_sn7_lbl(aoi, y, m):
    p = SN7_ROOT / 'train' / aoi / 'labels_raster' / \
        f'global_monthly_{y}_{m:02d}_mosaic_{aoi}_Buildings.tif'
    with rasterio.open(str(p)) as src:
        lbl = src.read(1)
    return (lbl > 0).astype(np.uint8)[CROP]

sn7_img_t1 = load_sn7_img(SN7_AOI, SN7_Y1, SN7_M1)
sn7_img_t2 = load_sn7_img(SN7_AOI, SN7_Y2, SN7_M2)
sn7_lbl_t1 = load_sn7_lbl(SN7_AOI, SN7_Y1, SN7_M1)
sn7_lbl_t2 = load_sn7_lbl(SN7_AOI, SN7_Y2, SN7_M2)

# Binary change map — matches downstream task: np.not_equal(building_t1, building_t2)
sn7_change = np.not_equal(sn7_lbl_t1, sn7_lbl_t2).astype(np.uint8)

sn7_rgb_t1 = to_rgb(sn7_img_t1, 0, 1, 2)
sn7_rgb_t2 = to_rgb(sn7_img_t2, 0, 1, 2)

print(f'Crop size: {sn7_rgb_t1.shape}')
print(f'Changed pixels: {sn7_change.sum()} / {sn7_change.size} ({100*sn7_change.mean():.1f}%)')

## PASTIS — Multi-temporal Crop Type Mapping

In [ ]:
PASTIS_ROOT = DATASET_ROOT / 'pastis'
PASTIS_ID   = '40047'   # 13 crop classes

# 19 semantic classes (index 19 = void / ignore)
PASTIS_CLASSES = [
    'Background', 'Meadow', 'Soft Winter Wheat', 'Corn', 'Winter Barley',
    'Winter Rapeseed', 'Spring Barley', 'Sunflower', 'Grapevine', 'Beet',
    'Winter Triticale', 'Winter Durum Wheat', 'Fruits/Veg./Flowers',
    'Potatoes', 'Leguminous Fodder', 'Soybeans', 'Orchard',
    'Mixed Cereal', 'Sorghum',
]
PASTIS_COLORS = [
    '#808080',  # 0  Background
    '#78c679',  # 1  Meadow
    '#ffffcc',  # 2  Soft Winter Wheat
    '#fdae61',  # 3  Corn
    '#d7191c',  # 4  Winter Barley
    '#ffd700',  # 5  Winter Rapeseed
    '#abd9e9',  # 6  Spring Barley
    '#ff8c00',  # 7  Sunflower
    '#7b2d8b',  # 8  Grapevine
    '#d01c8b',  # 9  Beet
    '#4dac26',  # 10 Winter Triticale
    '#b8e186',  # 11 Winter Durum Wheat
    '#2c7bb6',  # 12 Fruits/Veg./Flowers
    '#a6611a',  # 13 Potatoes
    '#80cdc1',  # 14 Leguminous Fodder
    '#018571',  # 15 Soybeans
    '#5e3c99',  # 16 Orchard
    '#e66101',  # 17 Mixed Cereal
    '#f4a582',  # 18 Sorghum
]

# S2: (T, 10, H, W) — bands [B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12]
s2  = np.load(PASTIS_ROOT / 'DATA_S2'     / f'S2_{PASTIS_ID}.npy').astype(np.float32)
ann = np.load(PASTIS_ROOT / 'ANNOTATIONS' / f'TARGET_{PASTIS_ID}.npy')[0]  # (H,W)

# Pick the time step with the highest mean NDVI (B8=idx6, B4=idx2)
ndvi_means = [(s2[t,6] - s2[t,2]).sum() / (s2[t,6] + s2[t,2] + 1e-6).sum()
              for t in range(s2.shape[0])]
best_t = int(np.argmax(ndvi_means))

# RGB: B4(idx2)=R, B3(idx1)=G, B2(idx0)=B
pastis_rgb     = to_rgb(s2[best_t], r=2, g=1, b=0)
pastis_gt      = np.ma.masked_where(ann == 19, ann)
pastis_present = sorted(set(np.unique(ann)) - {19})

# S1A: (71, 3, H, W) — channels: VV, VH, VV-VH (all in dB)
s1a = np.load(PASTIS_ROOT / 'DATA_S1A' / f'S1A_{PASTIS_ID}.npy').astype(np.float32)
best_s1 = min(best_t, s1a.shape[0] - 1)
vv = s1a[best_s1, 0]   # VV backscatter (dB)
vlo, vhi = np.percentile(vv, 2), np.percentile(vv, 98)
pastis_sar = np.clip((vv - vlo) / (vhi - vlo + 1e-6), 0, 1)   # grayscale in [0, 1]

print(f'S2  shape: {s2.shape},  best_t = {best_t}')
print(f'S1A shape: {s1a.shape}, best_s1 = {best_s1},  VV ∈ [{vv.min():.1f}, {vv.max():.1f}] dB')
print(f'Classes: {[PASTIS_CLASSES[c] for c in pastis_present]}')

## Combined figure

In [ ]:
# ── Layout ─────────────────────────────────────────────────────────────────
# 3 rows × 4 columns
#   Row 0  MADOS:       [S2 RGB (spans 2 cols)]  [GT mask]        [Legend]
#   Row 1  SpaceNet7:   [T1 RGB]  [T2 RGB]        [Binary change]  [hidden]
#   Row 2  PASTIS:      [S2 RGB]  [S1 SAR VV]     [Crop type map]  [Legend]

fig = plt.figure(figsize=(15, 11))
gs  = gridspec.GridSpec(
    3, 4, figure=fig,
    hspace=0.20, wspace=0.04,
    left=0.08, right=0.99, top=0.97, bottom=0.01,
    width_ratios=[1, 1, 1, 0.55],
)

# ── Row 0: MADOS ──────────────────────────────────────────────────────────
ax00 = fig.add_subplot(gs[0, 0:2])   # RGB spans first two columns
ax02 = fig.add_subplot(gs[0, 2])
ax03 = fig.add_subplot(gs[0, 3])

ax_img(ax00, mados_rgb, title='Sentinel-2 RGB')

mados_cmap, mados_norm = make_cmap(MADOS_COLORS)
mados_cmap.set_bad(color='black')    # ignored pixels → black
ax_mask(ax02, mados_gt_disp, mados_cmap, mados_norm, title='Semantic Labels')

legend_ax(ax03,
          [MADOS_CLASSES[c] for c in mados_present],
          [MADOS_COLORS[c]  for c in mados_present],
          fontsize=8)

# ── Row 1: SpaceNet7-CD ───────────────────────────────────────────────────
ax10 = fig.add_subplot(gs[1, 0])
ax11 = fig.add_subplot(gs[1, 1])
ax12 = fig.add_subplot(gs[1, 2])
ax13 = fig.add_subplot(gs[1, 3])
ax13.axis('off')

ax_img(ax10, sn7_rgb_t1, title=f'T₁ — {SN7_Y1}-{SN7_M1:02d}')
ax_img(ax11, sn7_rgb_t2, title=f'T₂ — {SN7_Y2}-{SN7_M2:02d}')

sn7_cmap, sn7_norm = make_cmap(['#d4d4d4', '#e41a1c'])
ax_mask(ax12, sn7_change, sn7_cmap, sn7_norm, title='Change Map')
sn7_patches = [
    mpatches.Patch(facecolor='#d4d4d4', edgecolor='0.4', lw=0.5, label='No Change'),
    mpatches.Patch(facecolor='#e41a1c', edgecolor='0.4', lw=0.5, label='Change'),
]
ax12.legend(handles=sn7_patches, loc='lower left', frameon=True,
            framealpha=0.85, edgecolor='0.7', fontsize=8,
            handlelength=1.1, handleheight=1.1, handletextpad=0.5,
            labelspacing=0.4)

# ── Row 2: PASTIS ─────────────────────────────────────────────────────────
ax20 = fig.add_subplot(gs[2, 0])
ax21 = fig.add_subplot(gs[2, 1])
ax22 = fig.add_subplot(gs[2, 2])
ax23 = fig.add_subplot(gs[2, 3])

ax_img(ax20, pastis_rgb, title=f'Sentinel-2 RGB (t={best_t})')

ax21.imshow(pastis_sar, cmap='gray', interpolation='nearest', vmin=0, vmax=1)
ax21.set_title(f'Sentinel-1 VV SAR (t={best_s1})', pad=4)
ax21.axis('off')

pastis_cmap, pastis_norm = make_cmap(PASTIS_COLORS)
ax_mask(ax22, pastis_gt, pastis_cmap, pastis_norm, title='Crop Type Labels')

legend_ax(ax23,
          [PASTIS_CLASSES[c] for c in pastis_present],
          [PASTIS_COLORS[c]  for c in pastis_present],
          fontsize=8)

# ── Row labels ────────────────────────────────────────────────────────────
ROW_META = [
    (ax00, 'MADOS',        '#1f77b4'),
    (ax10, 'SpaceNet7-CD', '#ff7f0e'),
    (ax20, 'PASTIS',       '#2ca02c'),
]
for ax, label, color in ROW_META:
    ax.set_ylabel(label, fontsize=12, fontweight='bold', color=color,
                  labelpad=8, rotation=90, va='center')

# ── Save ──────────────────────────────────────────────────────────────────
for fmt in ('png', 'pdf'):
    out = OUT_DIR / f'dataset_visualization.{fmt}'
    fig.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'Saved: {out}')

plt.show()